# Assignment 1: Fidelity - Financial Planning Agent
# Shishir Deshpande, MSDS 442

### Importing required libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

load_dotenv() # this will read my secrets and API keys from the .env file
os.environ["USER_AGENT"] = "MSDS442-Assignment1-Deshpande"

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
import fitz # this package has PyMuPDF tool
from IPython.display import display, Markdown

model = ChatOpenAI(model="gpt-4o-mini")

### Now I will load the Fidelity webpage

In [ ]:
url = 'https://www.fidelity.com/viewpoints/retirement/how-much-do-i-need-to-retire'

loader = WebBaseLoader(url) # this is the loader that will fetch the HTML in the URL and wrap it to make it ready for consumption
docs = loader.load() # This will return a list of objects to feed into the LLM

In [ ]:
# Confirming inputs are processed
print(f"Number of documents loaded: {len(docs)}") # this will confirm how many documents got loaded
print(f"Character count: {len(docs[0].page_content)}") # this will confirm how much content got loaded in the first document
print(docs[0].page_content[:1000]) # this will help me check what the content is about

In [ ]:
# Confirming if we have retirement related content in the doc
print("1x" in docs[0].page_content, "by 30" in docs[0].page_content)

Perfect! We have 1 document, 31K characters, and the first 1000 characters tell me its related to the content we are looking for. Second check confirms, we got retirement related content too. We can proceed now. 

### Splitting the text and creating the Chroma vector store

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
doc_splits = text_splitter.split_documents(docs)

# Confirm how many chunks this creates
print(f"Number of chunks created: {len(doc_splits)}")
print(doc_splits[0].page_content[:300])

# Creating the vector store
vectorstore = Chroma.from_documents(documents = doc_splits, embedding = OpenAIEmbeddings(model = "text-embedding-3-small"), collection_name="fidelity-retirement",)

retriever = vectorstore.as_retriever()

# Confirming the store was built
print("Vector store created successfully!")

### Setting up the retriever tool

In [ ]:
@tool # this will essentially make the entire function below a callable tool in LangChain
def retrieve_fidelity_docs(query: str) -> str:
    """Search and retrieve content that is relevant to answer retirement savings and planning questions. All content should be from Fidelity's retirement webpage."""
    results = retriever.invoke(query) # this will call the Chroma retriever's similarity search to return semantically similar chunks
    return "\n\n".join(doc.page_content for doc in results) # this will join all the retrieved chunks in spaced-out segments

# Performing a simple sanity check before we start working on the full graph
test_result = retrieve_fidelity_docs.invoke("How much should I save for retirement by age 30?")
print(test_result[:1000])

In [ ]:
prompt = PromptTemplate(
    template="""You are a financial advisor answering user question about retirement planning. \n
    Here is the retrieved document: \n\n {context} \n\n
    Here is the user question: {question} \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n 
    Give a binary score 'yes' or 'no' to indicate whether the document is relevant to answering the question.""", 
    input_variables=["context", "question"],)

In [ ]:
grader_chain = prompt | model

# Defining a function to to extract text content, change to all lower case and help the logic check for grading
def grade_relevance(question: str, context: str) -> str:
    result = grader_chain.invoke({"question": question, "context": context})
    return result.content.strip().lower()

# Checking for sanity before we wire it in
relevant_context = retrieve_fidelity_docs.invoke("How much should I save for retirement?")
irrelevant_context = retrieve_fidelity_docs.invoke("This document discusses Idli recipes.")

print("Relevant test:", grade_relevance("How much should I save for retirement?", relevant_context))
print("Irrelevant test:", grade_relevance("Why is the sky blue?", irrelevant_context))

Looks good! The relevance test does its job and catches my irrelevant question on why the sky is blue. 

### Setting up a multi-step LangGraph workflow to handle retrieval, relevance grading, and branching logic

In [ ]:
class GraphState(TypedDict): # Defining what the retrieve -> grade -> generate structure should look like
    messages: list
    context: str
    relevance: str

def retrieve_node(state: GraphState): # I want the question to be extracted from the last message
    question = state["messages"][-1][1]
    context = retrieve_fidelity_docs.invoke(question)
    return {"context": context}

def grade_node(state: GraphState):
    question = state["messages"][-1][1]
    relevance = grade_relevance(question, state["context"])
    return {"relevance": relevance}

def route_relevance(state: GraphState) -> Literal["generate", "no_answer"]:
    return "generate" if "yes" in state["relevance"] else "no_answer"

def generate_node(state: GraphState):
    question = state["messages"][-1][1]
    answer_prompt = f"""You are a Fidelity financial advisor. Using the context below, answer the user's question about retirement planning.

Context: {state['context']}
Question: {question}

Answer:"""
    response = model.invoke(answer_prompt)
    return {"messages": [response.content]}

def no_answer_node(state: GraphState): # This is what I would expect AI agent to respond to irrelevant questions
    return {"messages": ["I'm sorry, I can only answer questions related to retirement planning based on the Fidelity resources I have access to."]}

graph = StateGraph(GraphState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("grade", grade_node)
graph.add_node("generate", generate_node)
graph.add_node("no_answer", no_answer_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "grade")
graph.add_conditional_edges("grade", route_relevance, {"generate": "generate", "no_answer": "no_answer"})
graph.add_edge("generate", END)
graph.add_edge("no_answer", END)

app = graph.compile()

print("Graph has been compiled successfully.")

Whew! That was quite a bit of code but it all came out well. Graph is now built to handle our prompt in the way we want it. 

### Sending the required prompt to the graph

In [ ]:
inputs = {
    "messages": [
        ("user", "What are the steps that I should take to determine how much I need to save for retirement"),
    ]
}

result = app.invoke(inputs)
print(result)

In [ ]:
answer_text = result['messages'][0] if isinstance(result['messages'], list) else result['messages']
display(Markdown(f"### Relevance Grade: `{result['relevance']}`\n\n### Answer\n\n{answer_text}"))

Success! We got the message we needed that tells us what we need. But it looks a bit rough, so let's clean it up into a more readable format!

In [ ]:
test_prompts = [
    "What are the steps to take to determine my retirement savings?", # asks an on-topic question
    "How should I allocate my contributions between stocks, ETFs, and bonds?", # asks a semantically similar question
    "How do I make pancakes?", # asks an off-topic question
                ]

for prompt in test_prompts:
    test_input = {"messages": [("user", prompt)]}
    test_result = app.invoke(test_input)
    answer_text = test_result['messages'][0] if isinstance(test_result['messages'], list) else test_result['messages']
    display(Markdown(f"### Prompt: {prompt}\n\n**Relevance Grade:** `{test_result['relevance']}`\n\n**Response:**\n\n{answer_text}\n\n---"))

Perfect! It did not get tripped up when I asked it how to make pancakes. But it gave me detailed responses on how to allocate my contributions and how to determine my retirement savings.

Much better now. The results are far more readable for use and verification. The AI agent rated the question as relevant, then proceeded to generate a full-form answer to the question. It clearly denotes the steps in a bullet-point format, adds recommendations such as saving 15% of your income annually, and provides considerations like personal factors. This tells me that it is providing us with solid financial advice akin to a Fidelity financial advisor. 

### Extracting the text from Fidelity Freedom® 2045 Fund's (FFFGX) fact sheet

In [ ]:
pdf_path = "FFFGX_FactSheet.pdf"

doc = fitz.open(pdf_path)
num_pages = len(doc)

full_text = ""

for page in doc:
    full_text += page.get_text()

doc.close()

# Verifying the extraction is appropriate
print(f"Number of pages: {num_pages}")
print(f"Character count: {len(full_text)}")
print(full_text[:1000])

In [ ]:
# Verifying content is good for AI agent interaction
print("Freedom 2045" in full_text, "Expense Ratio" in full_text.replace("\n", " "))

Good. We have confirmation that the PDF extraction is working as intended. I used `full_text.replace("\n", " ")` to make sure line split did not prevent our checks.

### Defining required questions for our chat agent

In [ ]:
questions = [
        "What is the name of this fund?",
        "Who is the fund manager?",
        "What is the calendar year return for 2022 for this fund and S&P 500?",
        "What is the Portfolio Net Assets?",
        "What is the Morningstar rating for this fund? How many funds were used to rate this fund?",
]

questions_block = "\n".join(f"{i+1}. {q}" for i, q in enumerate(questions))
print(questions_block)

### Sending all the extracted information from the fact sheet PDF and the questions to the OpenAI LLM

In [ ]:
system_prompt = SystemMessage(content = "You are a financial analyst's assistant. You will need to answer questions about this fund using ONLY the fact sheet content that has been provided. If the fact sheet does not contain the answer to a question, you should respond that you do not have the answer instead of guessing.")

human_prompt = HumanMessage(content=f"""Fact Sheet Content:
                            {full_text}

Please answer the following questions based on the fact sheet above:

{questions_block}""")

response = model.invoke([system_prompt, human_prompt])
display(Markdown(f"### Fidelity Freedom 2045 Fund - FFFGX Q&A\n\n{response.content}"))

The output is nearly exactly what I expected it to be. However, the AI model is unable to answer question 5 (`5. What is the Morningstar rating for this fund? How many funds were used to rate this fund?`) as the star rating is stored as an image in the PDF fact sheet. The fitz/PyMuPDF library only extracted textual content, so the star rating never reached our AI agent's context. 

I manually verified the rating (`5 stars out of 181 funds`) and noted that the model does not overreach or hallucinate the response. In fact, it tried to be helpful and stated what it can share in absence of the star rating. I am treating this as an intended consequence of the model/tool's implementation. This is quite realistically (and frustratingly, sometimes) a real-world limitation of working with PDF documents, as they are so friendly for storing textual and graphic content in the same document so seamlessly.

![image.png](attachment:image.png)